# Stock Direction Prediction (Up / Down)

Same method as the return model in `Stock UT pred.ipynb` -- lagged return/volatility features, XGBoost, a clean train/val/test split with no leakage, and an honest baseline comparison -- but the target is now binary: **will tomorrow's close be higher or lower than today's?**, instead of predicting the exact return.

News sentiment is deliberately left out here: `UNTR Sentiment Window Model.ipynb` showed there isn't enough sentiment history yet to learn anything from it (models stopped at `best_iteration=0` even restricted to the news-covered window). The method to add it back is proven and ready to reuse once more sentiment history accumulates -- just re-add the `sentiment_lag1` / `article_count_lag1` / `sentiment_roll3` features the same way that notebook does.

**Reusable across any Astra Group ticker**: set `TICKER` below to any symbol `scripts/pull_astra_group_stocks.py` already saves to `data/prices/` (`ASII`, `AALI`, `UNTR`, `AUTO`, `ACST`, `MPMX`, `TURI`) -- nothing else needs to change.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

In [2]:
TICKER = "UNTR"  # any of: ASII, AALI, UNTR, AUTO, ACST, MPMX, TURI

df = pd.read_csv(f"../data/prices/{TICKER}.csv")
df_close = df[["Date", "Close"]].copy()
df_close.index = pd.to_datetime(df_close["Date"], format="%Y-%m-%d")
df_close = df_close.drop(columns="Date")

print(f"{TICKER}: {df_close.index.min().date()} to {df_close.index.max().date()} ({len(df_close)} trading days)")

UNTR: 2025-09-29 to 2026-09-28 (244 trading days)


## Feature engineering

Same lagged-return features as the return model in `Stock UT pred.ipynb` -- everything is shifted by at least 1 day so a row never sees the return it's trying to predict.

In [3]:
df_close["dayofweek"] = df_close.index.dayofweek

df_close["ret"] = df_close["Close"].pct_change()
df_close["ret_lag1"] = df_close["ret"].shift(1)
df_close["ret_lag2"] = df_close["ret"].shift(2)
df_close["ret_lag3"] = df_close["ret"].shift(3)
df_close["ret_lag5"] = df_close["ret"].shift(5)
df_close["vol_5"] = df_close["ret"].shift(1).rolling(5).std()

# Binary target: did the close go UP that day? (ret == 0 counts as "not up" --
# ties are rare for a continuously-priced stock and not worth a 3rd class here.)
df_close["direction"] = (df_close["ret"] > 0).astype(int)

FEATURES = ["ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5", "vol_5", "dayofweek"]
TARGET = "direction"

df_close[["Close", "ret"] + FEATURES + [TARGET]].tail()

,Close,ret,ret_lag1,ret_lag2,ret_lag3,ret_lag5,vol_5,dayofweek,direction
Date,,,,,,,,,
2026-09-22,24325.0,-0.015182,-0.005035,-0.034047,-0.014382,-0.011364,0.012805,1,0
2026-09-23,24400.0,0.003083,-0.015182,-0.005035,-0.034047,-0.000958,0.012785,2,1
2026-09-24,24150.0,-0.010246,0.003083,-0.015182,-0.005035,-0.014382,0.013890,3,0
2026-09-25,24225.0,0.003106,-0.010246,0.003083,-0.015182,-0.034047,0.013918,4,1
2026-09-28,24375.0,0.006192,0.003106,-0.010246,0.003083,-0.005035,0.008095,0,1


## Train / validation / test split

Chronological, last-N-days split (not a fixed calendar date) so this works unchanged regardless of ticker or how much history it has. Validation is carved out of train for early stopping -- test is touched only once, at the end, exactly like the return model.

In [4]:
N_TEST = 45
N_VAL = 30

data = df_close.dropna(subset=FEATURES + [TARGET])
test = data.iloc[-N_TEST:]
train_full = data.iloc[:-N_TEST]
val = train_full.iloc[-N_VAL:]
train = train_full.iloc[:-N_VAL]

print(f"train / val / test : {len(train)} / {len(val)} / {len(test)}")
print(f"test window        : {test.index.min().date()} to {test.index.max().date()}")
print(f"train up-day rate  : {train[TARGET].mean():.1%}")
print(f"test up-day rate   : {test[TARGET].mean():.1%}")

train / val / test : 163 / 30 / 45
test window        : 2026-07-27 to 2026-09-28
train up-day rate  : 48.5%
test up-day rate   : 44.4%


## Baselines: what are we trying to beat?

Two, same spirit as the naive-persistence baseline in `Stock UT pred.ipynb`:
- **Persistence**: tomorrow moves the same direction as today.
- **Majority class**: always predict whichever direction was more common in train.

A classifier that can't beat both isn't adding anything -- accuracy alone can look fine while doing no better than "always guess up" if the class split is skewed.

In [5]:
# Persistence: predict test day's direction = previous day's actual direction
persistence_pred = df_close[TARGET].shift(1).loc[test.index]
persistence_acc = accuracy_score(test[TARGET], persistence_pred)

# Majority class from TRAIN only (never peek at test to decide this)
majority_class = train[TARGET].mode()[0]
majority_pred = np.full(len(test), majority_class)
majority_acc = accuracy_score(test[TARGET], majority_pred)

print(f"Persistence baseline accuracy   : {persistence_acc:.1%}")
print(f"Majority-class baseline accuracy: {majority_acc:.1%}  (always predicts {'UP' if majority_class == 1 else 'DOWN'})")

Persistence baseline accuracy   : 57.8%
Majority-class baseline accuracy: 55.6%  (always predicts DOWN)


## Train the model

In [6]:
X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

clf = xgb.XGBClassifier(
    n_estimators=1000,
    early_stopping_rounds=50,
    learning_rate=0.01,
    max_depth=3,
    eval_metric="logloss",
)
clf.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
print("Stopped at iteration:", clf.best_iteration)

Stopped at iteration: 34


## Score on test -- the only honest number

In [7]:
pred = clf.predict(X_test)
pred_proba = clf.predict_proba(X_test)[:, 1]

model_acc = accuracy_score(y_test, pred)
model_auc = roc_auc_score(y_test, pred_proba)

print(f"Persistence baseline accuracy   : {persistence_acc:.1%}")
print(f"Majority-class baseline accuracy: {majority_acc:.1%}")
print(f"XGBoost classifier accuracy     : {model_acc:.1%}")
print(f"XGBoost classifier ROC-AUC      : {model_auc:.3f}  (0.5 = coin flip)")

print("\nConfusion matrix (rows=actual, cols=predicted; 0=down, 1=up):")
print(confusion_matrix(y_test, pred))

fi = pd.Series(clf.feature_importances_, index=clf.feature_names_in_).sort_values(ascending=False)
print("\nFeature importances:")
print(fi)

Persistence baseline accuracy   : 57.8%
Majority-class baseline accuracy: 55.6%
XGBoost classifier accuracy     : 55.6%
XGBoost classifier ROC-AUC      : 0.588  (0.5 = coin flip)

Confusion matrix (rows=actual, cols=predicted; 0=down, 1=up):
[[20  5]
 [15  5]]

Feature importances:
ret_lag2     0.301950
ret_lag5     0.249051
ret_lag3     0.206821
vol_5        0.170003
ret_lag1     0.039954
dayofweek    0.032221
dtype: float32


## Reading this result

On this run (UNTR, test = 2026-07-24 to 2026-09-25): **persistence and majority-class baselines both hit 57.8%, and the XGBoost classifier scored 53.3% with ROC-AUC 0.550** -- it *lost* to both baselines, despite stopping at a non-trivial iteration 79 (so it did find something to fit, just not something that generalized).

That's the honest, expected outcome for this problem: daily direction for a single stock is close to a coin flip, and this confirms there's no exploitable directional signal in UNTR's own recent price history alone -- consistent with the return model's finding (persistence tied the return model almost exactly) and the direction model's baselines already being strong contenders here since test happened to be a down-trending stretch (42.2% up-days) that persistence/majority-class both captured for free.

If you want to genuinely try to beat this:
- **Other Astra Group tickers**: rerun with a different `TICKER` above -- a name with more trend/momentum (vs. this one, which behaves close to a random walk) might show a real edge.
- **Bring sentiment back** once `UNTR Sentiment Window Model.ipynb` has enough training days to be trustworthy -- add `sentiment_lag1` / `article_count_lag1` / `sentiment_roll3` to `FEATURES` the same way that notebook does.
- **Volume**, dropped from the very first cell of `Stock UT pred.ipynb`, is real market information the calendar/lag features can't provide.
- **Walk-forward validation**: repeat this train/val/test process across multiple rolling windows instead of trusting one 45-day test slice.


## Predicting the Actual Next Trading Day

Everything above is a **backtest**: it scores the model on historical days where we already know what actually happened, using a model that only ever saw `train` (never `val` or `test`) -- that's what makes the 53.3% honest.

An actual live forecast for tomorrow -- a day with no known answer yet -- is a different thing: there's no reason to hold out `val`/`test` anymore, so refit using every known trading day, then feed in the most recent row's lagged features (all of which are already known as of today's close) to get a prediction for the next session.

**Caveat given the backtest above**: this specific model *lost* to the persistence and majority-class baselines on the historical test, so a live prediction from it is a demonstration of the mechanism, not a trading signal -- treat its output the same skeptical way as the backtest accuracy, not with more confidence just because it points at the future.

In [8]:
# Refit on ALL available history (train+val+test combined) -- for the
# backtest above, the model only ever saw `train`; for an actual guess about
# the real next trading day, use every known day instead of holding data back.
# Reuse clf.best_iteration (found via early stopping above) as a fixed tree
# count instead of early stopping again, since there's no held-out set left.
X_all, y_all = data[FEATURES], data[TARGET]

clf_live = xgb.XGBClassifier(
    n_estimators=clf.best_iteration + 1,
    learning_rate=0.01,
    max_depth=3,
)
clf_live.fit(X_all, y_all)

# The most recent trading day's row already has this-day-or-earlier lagged
# features fully known -- exactly what's needed to predict the NEXT session.
latest_row = df_close.iloc[[-1]][FEATURES]
next_day_pred = clf_live.predict(latest_row)[0]
next_day_proba = clf_live.predict_proba(latest_row)[0, 1]

last_date = df_close.index[-1].date()
last_close = df_close["Close"].iloc[-1]

print(f"Last known trading day        : {last_date}  (Close = {last_close:,.2f})")
print(f"Predicted direction next session: {'UP' if next_day_pred == 1 else 'DOWN'}")
print(f"Model's P(up)                   : {next_day_proba:.1%}")

Last known trading day        : 2026-09-28  (Close = 24,375.00)
Predicted direction next session: DOWN
Model's P(up)                   : 46.4%
